# Clustering — Stroke MERSCOPE (Fan / CG)

Unsupervised clustering of the Baysor-segmented cells produced by `baysor_stroke_batch.ipynb` →
`baysor_stroke_to_h5ad.ipynb`.

Input: `stroke_all.h5ad` (cells × 500-gene panel, **raw integer counts** in `X`, Baysor per-cell QC in `.obs`,
centroids in `.obsm['spatial']`, region id in `.obs['sample']`).

Pipeline:
1. QC + filtering (drop low-count / low-gene cells).
2. Normalize (total-count → log1p), keep raw counts in `layers['counts']`.
3. PCA → neighbors → Leiden → UMAP. (Optional Harmony batch correction across regions.)
4. Marker genes per cluster, UMAP + spatial plots.
5. Save `stroke_all_clustered.h5ad`.

**Kernel:** `sc`. This is a **targeted 500-gene panel**, so we cluster on *all* genes (no HVG selection).

> Re-run safe: nothing here writes back to the Baysor outputs; only a new `*_clustered.h5ad` is produced.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from pathlib import Path

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")

h5ad_dir = Path("/Volumes/T7/Stroke_merscop_Fan_CG/h5ad")
adata = sc.read_h5ad(h5ad_dir / "stroke_all.h5ad")


# Brain-only: drop the two spinal cord (SC) samples — they are clustered separately
SC_SAMPLES = ["SC_uninjured_1", "SC_uninjured_2"]
n_before = adata.n_obs
adata = adata[~adata.obs["sample"].isin(SC_SAMPLES)].copy()
print(f"Dropped SC samples: {n_before} -> {adata.n_obs} cells "
      f"({n_before - adata.n_obs} removed across {len(SC_SAMPLES)} SC regions)")
adata.obs["sample"] = adata.obs["sample"].astype("category")
print(adata)
print(adata.obs["sample"].value_counts())

## 1. QC

Imaging-based panels are small, so cells carry few transcripts (median ~33 counts here). We look at the
distributions, then drop cells that are too sparse to type reliably and genes seen in almost no cell.

In [ ]:
adata.var_names_make_unique()
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].hist(adata.obs["total_counts"], bins=100); axs[0].set_xlabel("total_counts"); axs[0].set_yscale("log")
axs[1].hist(adata.obs["n_genes_by_counts"], bins=100); axs[1].set_xlabel("n_genes_by_counts")
if "area" in adata.obs:
    axs[2].hist(adata.obs["area"], bins=100); axs[2].set_xlabel("area"); axs[2].set_yscale("log")
plt.tight_layout(); plt.show()

adata.obs[["total_counts", "n_genes_by_counts"]].describe()

In [ ]:
# --- filtering thresholds (tune to the histograms above) ---
MIN_COUNTS = 10     # min transcripts per cell
MIN_GENES  = 2     # min distinct genes per cell
MIN_CELLS  = 10     # min cells expressing a gene to keep it

n0 = adata.n_obs
sc.pp.filter_cells(adata, min_counts=MIN_COUNTS)
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
print(f"cells: {n0} -> {adata.n_obs}  ({n0 - adata.n_obs} removed)")
print(f"genes kept: {adata.n_vars}")

## 2. Normalize

Keep raw counts in `layers['counts']` (needed later for marker tests / export), then total-count normalize
and log1p.

In [ ]:
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata)        # median-count normalization
sc.pp.log1p(adata)
adata.raw = adata                    # log-normalized snapshot for plotting / DE

# Targeted panel: use all genes. Scale (clip outliers) before PCA.
#sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50)
sc.pl.pca_variance_ratio(adata, n_pcs=50)

## 3. (Optional) batch correction across regions

Three regions so far — if clusters separate by `sample` rather than biology, run Harmony on the PCA embedding.
Leave this cell unrun to cluster on the raw PCA. (`harmonypy` is pip-installed on first use.)

In [ ]:
USE_HARMONY = False   # set True to batch-correct on 'sample'

rep = "X_pca"
if USE_HARMONY:
    try:
        import harmonypy  # noqa: F401
    except ModuleNotFoundError:
        import sys, subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "harmonypy"])
    sc.external.pp.harmony_integrate(adata, key="sample")
    rep = "X_pca_harmony"
print("Using representation:", rep)

## 4. Neighbors → Leiden → UMAP

In [ ]:
RESOLUTION = 1.0    # higher -> more clusters

sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, use_rep=rep)
sc.tl.leiden(adata, resolution=RESOLUTION, key_added="leiden")
sc.tl.umap(adata)
print("n clusters:", adata.obs["leiden"].nunique())
adata.obs["leiden"].value_counts().sort_index()

In [ ]:
sc.pl.umap(adata, color=["leiden", "sample"], wspace=0.35)
sc.pl.umap(adata, color=["total_counts", "n_genes_by_counts"], wspace=0.35)

## 5. Marker genes per cluster

Wilcoxon rank-sum on the log-normalized data (`.raw`) to find each cluster's top genes — the starting point
for annotating cell types.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon", use_raw=True)
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False, fontsize=8)

top = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(10)
top

In [ ]:
# Dotplot of the top markers per cluster
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, use_raw=True)

## 6. Spatial map of clusters

Plot each region in its physical coordinates, colored by cluster, to check the clusters form coherent
anatomy (and to eyeball the lesion).

In [ ]:
samples = adata.obs["sample"].cat.categories if hasattr(adata.obs["sample"], "cat") \
    else sorted(adata.obs["sample"].unique())
n = len(samples)
ncol = min(3, n)
nrow = int(np.ceil(n / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(6 * ncol, 6 * nrow), squeeze=False)

palette = sc.pl.palettes.default_20
clusters = list(adata.obs["leiden"].cat.categories)
color_map = {c: palette[i % len(palette)] for i, c in enumerate(clusters)}

for ax, s in zip(axs.ravel(), samples):
    sub = adata[adata.obs["sample"] == s]
    xy = sub.obsm["spatial"]
    cols = sub.obs["leiden"].map(color_map).values
    ax.scatter(xy[:, 0], xy[:, 1], c=cols, s=2, linewidths=0)
    ax.set_title(s, fontsize=10); ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])
for ax in axs.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# --- Split sample names into metadata columns (for KaroSpace) ---------------
# Layout: <animal_id>_<line>_<model>[_<treatment>][_<timepoint>][_<replicate>]
# Irregular: some samples lack a timepoint and/or replicate; a few carry a
# treatment token (clp / lip). Parsed positionally with those slots optional.
import re

def parse_sample(name):
    tok = name.split("_")
    animal_id = tok[0]
    line      = tok[1] if len(tok) > 1 else None
    rest      = tok[2:]

    # trailing pure-integer token -> replicate
    replicate = None
    if rest and rest[-1].isdigit():
        replicate, rest = rest[-1], rest[:-1]

    # token like 5d / 7d / 14d -> timepoint; everything else is model/treatment
    timepoint, body = None, []
    for t in rest:
        if re.fullmatch(r"\d+d", t):
            timepoint = t
        else:
            body.append(t)

    model     = body[0] if body else None
    treatment = "_".join(body[1:]) if len(body) > 1 else None

    # Normalize the stroke model. dmcao/dcmao -> dMCAO, tmcao/tcmao -> tMCAO
    # (the 'c'/'m' transpositions are spelling variants of the same surgery).
    m = (model or "").lower()
    if re.fullmatch(r"[dt]c?mc?ao", m):          # matches dmcao/dcmao/tmcao/tcmao
        model_norm = "dMCAO" if m.startswith("d") else "tMCAO"
    else:
        model_norm = model

    injured = "uninjured" if m == "uninjured" else "injured"
    tp_days = int(timepoint[:-1]) if timepoint else (0 if injured == "uninjured" else np.nan)
    condition = "_".join(b for b in (model_norm, treatment, timepoint) if b)

    return dict(animal_id=animal_id, line=line, model=model, model_norm=model_norm,
                treatment=treatment or "none", timepoint=timepoint or "none",
                timepoint_days=tp_days, replicate=replicate or "NA",
                injured=injured, condition=condition)

meta = pd.DataFrame({s: parse_sample(s) for s in adata.obs["sample"].unique()}).T
meta.index.name = "sample"

# write back onto cells (string fields categorical, timepoint_days numeric)
for col in meta.columns:
    mapped = adata.obs["sample"].map(meta[col])
    adata.obs[col] = pd.to_numeric(mapped) if col == "timepoint_days" else mapped.astype("category")

print("Added obs columns:", list(meta.columns))
meta

## 7. Save

In [ ]:
adata.obs

In [ ]:
out_path = h5ad_dir / "stroke_all_clustered.h5ad"
adata.write_h5ad(out_path)
print("Wrote", out_path)
print(adata)